<a href="https://colab.research.google.com/github/cchen744/olist-regional-customer-experience-analysis/blob/main/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Library import

In [5]:
import os
import pandas as pd
import geopandas
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from google.colab import userdata

# Load datasets
Since it is not super convenient to merge all the dataset into one universal dataset (as it would be extremely big and hard to process), I am going to focus on subdatasets serving for seperate purposes (delievery performance, spatial features, order & product features).


In [2]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME') # Replace with username from json
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')           # Replace with key from json

from kaggle.api.kaggle_api_extended import KaggleApi

# 1. Authenticate
api = KaggleApi()
api.authenticate()

# 2. Download and Extract
# This handles the "extraction" part automatically with unzip=True
dataset_slug = "olistbr/brazilian-ecommerce"
download_path = "./brazilian_ecommerce_data"

print(f"Downloading {dataset_slug}...")
api.dataset_download_files(dataset_slug, path=download_path, unzip=True)
print("Download and extraction complete.\n")

# 3. Load the data using os.listdir
# The Olist dataset has multiple CSV files (orders, customers, products, etc.)
dataframes = {}

print("Loading files...")
for filename in os.listdir(download_path):
    if filename.endswith(".csv"):
        # Construct full file path
        file_path = os.path.join(download_path, filename)

        # Create a simpler name for the dataframe (e.g., "olist_orders_dataset.csv" -> "orders")
        # This is optional but makes accessing them easier
        df_name = filename.split('_dataset')[0].replace('olist_', '')

        # Load into dictionary
        dataframes[df_name] = pd.read_csv(file_path)
        print(f"--> Loaded {filename} as '{df_name}' (Shape: {dataframes[df_name].shape})")

# Example: Accessing one of the loaded tables
print("\nExample: First 5 rows of orders:")

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
Download and extraction complete.

Loading files...
--> Loaded olist_products_dataset.csv as 'products' (Shape: (32951, 9))
--> Loaded olist_customers_dataset.csv as 'customers' (Shape: (99441, 5))
--> Loaded olist_order_payments_dataset.csv as 'order_payments' (Shape: (103886, 5))
--> Loaded olist_orders_dataset.csv as 'orders' (Shape: (99441, 8))
--> Loaded olist_order_reviews_dataset.csv as 'order_reviews' (Shape: (99224, 7))
--> Loaded olist_sellers_dataset.csv as 'sellers' (Shape: (3095, 4))
--> Loaded product_category_name_translation.csv as 'product_category_name_translation.csv' (Shape: (71, 2))
--> Loaded olist_order_items_dataset.csv as 'order_items' (Shape: (112650, 7))
--> Loaded olist_geolocation_dataset.csv as 'geolocation' (Shape: (1000163, 5))

Example: First 5 rows of orders:


In [3]:
for df_name, df in dataframes.items():
    print('\n',df_name,'\n', df.columns,'\n',df.shape)


 products 
 Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object') 
 (32951, 9)

 customers 
 Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object') 
 (99441, 5)

 order_payments 
 Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object') 
 (103886, 5)

 orders 
 Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object') 
 (99441, 8)

 order_reviews 
 Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
    

## The join table


| From Table  | Key Column      | To Table       | Key Column      | Relationship | Purpose                   |
| ----------- | --------------- | -------------- | --------------- | ------------ | ------------------------- |
| orders      | customer_id     | customers      | customer_id     | many → one   | get customer location     |
| orders      | order_id        | order_reviews  | order_id        | one → one    | get review_score (target) |
| orders      | order_id        | order_items    | order_id        | one → many   | get product & seller info |
| order_items | product_id      | products       | product_id      | many → one   | get product attributes    |
| order_items | seller_id       | sellers        | seller_id       | many → one   | get seller location       |
| orders      | order_id        | order_payments | order_id        | one → many   | get payment info          |
| customers   | zip_code_prefix | geolocation    | zip_code_prefix | many → many  | get lat/lng (optional)    |
| sellers     | zip_code_prefix | geolocation    | zip_code_prefix | many → many  | get lat/lng (optional)    |


In [16]:
orders = dataframes['orders'].copy()

orders = orders.dropna(subset=[
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

order_reviews = dataframes['order_reviews'][['order_id', 'review_score']].copy()
order_reviews = order_reviews.copy()
order_reviews.loc[:, 'negative_review'] = order_reviews['review_score'] <= 2

order_items = dataframes['order_items'].dropna()
products = dataframes['products'].dropna()
sellers = dataframes['sellers'].dropna()
order_payments = dataframes['order_payments'].dropna()
customers = dataframes['customers'].dropna()
geolocation = dataframes['geolocation'].dropna()

## Features for delivery performance

Tables of interests
- orders
- order_reviews

Features to be created
- delivery_days # delievered_time - ordered_time
- late_delievery # wether an order is late delievered
- delay_days # how long an order is delayed

In [15]:
delievery_performance = orders.merge(order_reviews, on='order_id')
print(delievery_performance.columns)
print(delievery_performance.shape)
print(delievery_performance.head(1))

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'review_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp', 'negative_review'],
      dtype='object')
(9500, 15)
                           order_id                       customer_id  \
0  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2018-07-26 14:31:00           2018-08-07 15:27:45   

  order_estimated_delivery_date                         review_id  \
0           2018-08-13 00:00:00  8d5266042046a06655c8db133d120ba5   

   review_score review_comment_title revi

In [ ]:
# convert time data for convenience of duration computation
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    delievery_performance[col] = pd.to_datetime(delievery_performance[col])

In [ ]:
delievery_performance['delivery_days']=delievery_performance['order_delivered_customer_date']-delievery_performance['order_purchase_timestamp']
delievery_performance['delivery_days'] = (
    delievery_performance['order_delivered_customer_date'] -
    delievery_performance['order_purchase_timestamp']
).dt.days

delievery_performance['delay_days'] = (
    delievery_performance['order_delivered_customer_date'] -
    delievery_performance['order_estimated_delivery_date']
).dt.days

delievery_performance['late_delivery'] = (
    delievery_performance['delay_days'] > 0
)